In [1]:
import gymnasium as gym
from dataclasses import dataclass, asdict

import torch as t
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

import numpy as np

In [2]:
env = gym.make('CartPole-v1')

assert isinstance(env.action_space, gym.spaces.Discrete)

n_action = int(env.action_space.n)
obs_dim = env.observation_space.shape

In [3]:
device = "cuda"
n_episodes = 5_000
batch_size = 64

In [4]:
# policy takes in state, outputs action distribution
policy = nn.Sequential(
  nn.Linear(obs_dim[-1], 64),
  nn.ReLU(),
  nn.Linear(64, n_action),
  nn.Softmax(dim=-1)
).to(device)

 
df = 0.99  # discount factor gamma
lr = 3e-3  # this is alpha = step_size
optim = t.optim.Adam(policy.parameters(), lr=lr)

In [5]:

for ep in tqdm(range(n_episodes)):
  # generate episode
  state, _ = env.reset()
  state = t.tensor(state, dtype=t.float, device=device)
  done = False

  actions, states, rewards = [], [], []
  while not done:
    action_probs = policy(state)
    dist = t.distributions.Categorical(probs=action_probs)
    action = dist.sample().item()

    state_next, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    # add to buffer
    actions.append(action)
    states.append(state)
    rewards.append(reward)

    state = t.tensor(state_next, dtype=t.float, device=device)

  # compute discounted returns
  discounted_returns = []  # G values

  for t_step in range(len(rewards)):
    G = 0.0
    for k, r in enumerate(rewards[t_step:]):
      G += (df**k)*r
    discounted_returns.append(G)

  # normalize returns to reduce variance
  discounted_returns = t.tensor(discounted_returns, dtype=t.float, device=device)
  discounted_returns = (discounted_returns - discounted_returns.mean()) / (discounted_returns.std() + 1e-8)

  # update policy
  optim.zero_grad()
  loss = t.zeros((), device=device)
  for state, action, G in zip(states, actions, discounted_returns):
    action_probs = policy(state)
    dist = t.distributions.Categorical(action_probs)
    log_prob = dist.log_prob(t.tensor(action, device=device))

    loss = loss - log_prob*G  # we multiply by -1 since gradient descent is designed to decrease loss, but we want to increase the log_prob

  loss.backward()
  optim.step()


100%|██████████| 5000/5000 [41:22<00:00,  2.01it/s]


In [8]:
# execute policy
eval_env = gym.make('CartPole-v1', render_mode='human')

state, _ = eval_env.reset()
state = t.tensor(state, dtype=t.float, device=device)
done = False

while not done:
  action_probs = policy(state)
  dist = t.distributions.Categorical(probs=action_probs)
  action = dist.sample().item()

  state_next, reward, terminated, truncated, _ = eval_env.step(action)
  done = terminated or truncated

  state = t.tensor(state_next, dtype=t.float, device=device)

eval_env.close()